# jev-my-bro — Colab training\n\nEnable a GPU in **Runtime → Change runtime type → GPU**. Put this repository at `/content/jev-my-bro` (upload/copy/clone it), then run all cells. The dataset is already checked into the repository; this notebook does not generate training data.

In [ ]:
!nvidia-smi\nimport torch\nprint('cuda:', torch.cuda.is_available())\nif torch.cuda.is_available():\n    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path\nimport os\nREPO = Path('/content/jev-my-bro')\nassert (REPO / 'requirements.txt').exists(), 'Place the repo at /content/jev-my-bro first'\nos.chdir(REPO)\nprint('repo:', Path.cwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!python scripts/validate_dataset.py

In [ ]:
!python -m training.train --train data/train.jsonl --validation data/validation.jsonl --output artifacts/model --epochs 4 --train-batch-size 16 --eval-batch-size 32

In [ ]:
!python -m training.calibrate --model artifacts/model --data data/calibration.jsonl --output artifacts/calibration.json

In [ ]:
!python -m training.evaluate --model artifacts/model --data data/test.jsonl --calibration artifacts/calibration.json

In [ ]:
!python -m export.export_onnx --model artifacts/model --output artifacts/onnx/model.onnx

## Persist the trained artifacts\n\nColab storage is temporary. The final cell copies the model, calibration file, and ONNX file into Google Drive.

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n!mkdir -p /content/drive/MyDrive/jev-my-bro-artifacts\n!cp -r artifacts/model /content/drive/MyDrive/jev-my-bro-artifacts/\n!cp artifacts/calibration.json /content/drive/MyDrive/jev-my-bro-artifacts/\n!mkdir -p /content/drive/MyDrive/jev-my-bro-artifacts/onnx\n!cp artifacts/onnx/model.onnx /content/drive/MyDrive/jev-my-bro-artifacts/onnx/\nprint('saved to Google Drive: MyDrive/jev-my-bro-artifacts')